# 09 — `nn.Module` in Depth

In the previous notebook, we learned the foundations of neural networks and built simple models using:

- `nn.Linear`
- Activation functions
- `nn.Module`
- `nn.Sequential`

Now we will study `nn.Module` much more deeply.

`nn.Module` is the base class used to build almost every PyTorch neural network.

It helps PyTorch automatically manage:

- Learnable parameters
- Submodules
- Training and evaluation modes
- Device movement
- Saving and loading
- Model organization

## In this notebook, we will learn:

1. The `nn.Module` base class
2. `__init__()`
3. `forward()`
4. Parameter registration
5. `model.parameters()`
6. `model.named_parameters()`
7. `state_dict()`
8. Submodules
9. `nn.Sequential`
10. Custom layers
11. Training mode vs evaluation mode
12. Saving and loading model weights
13. Building reusable PyTorch model classes
14. Common mistakes
15. Debugging model structure
16. Practice exercises

## Main Goal

By the end of this notebook, you should understand how PyTorch organizes a model internally.

The most important mental model is:

$$
\boxed{
\text{Model}
=
\text{Submodules}
+
\text{Registered Parameters}
+
\text{Forward Logic}
}
$$


In [ ]:
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)


# 1. What Is `nn.Module`?

`nn.Module` is the base class for PyTorch models and layers.

When we write:

```python
class MyModel(nn.Module):
```

we are telling PyTorch:

> This class represents a model or layer that may contain parameters and other modules.

Examples of built-in PyTorch modules include:

- `nn.Linear`
- `nn.Conv2d`
- `nn.ReLU`
- `nn.Dropout`
- `nn.BatchNorm1d`
- `nn.Sequential`

Even these built-in layers inherit from `nn.Module`.


In [ ]:
print(issubclass(nn.Linear, nn.Module))
print(issubclass(nn.ReLU, nn.Module))
print(issubclass(nn.Sequential, nn.Module))


# 2. A Minimal Custom Model

Let's create a small model.

Architecture:

$$
3 \rightarrow 4 \rightarrow 2
$$


In [ ]:
class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(3, 4)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(4, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = TinyModel()

print(model)


The model contains three submodules:

1. `fc1`
2. `relu`
3. `fc2`

The model itself is also an `nn.Module`.


In [ ]:
print(type(model))
print(isinstance(model, nn.Module))


# 3. Understanding `__init__()`

The `__init__()` method defines the model's components.

In our model:

```python
self.fc1 = nn.Linear(3, 4)
self.relu = nn.ReLU()
self.fc2 = nn.Linear(4, 2)
```

These assignments are important.

Because `fc1`, `relu`, and `fc2` are assigned as attributes of an `nn.Module`, PyTorch automatically registers them as submodules.

That registration allows PyTorch to find their parameters later.


# 4. Why `super().__init__()` Is Important

Inside a custom model, we usually write:

```python
super().__init__()
```

This initializes the parent `nn.Module` class.

Without it, important internal machinery for:

- Module registration
- Parameter registration
- Hooks
- State management

may not be initialized correctly.


In [ ]:
class CorrectModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(2, 1)

    def forward(self, x):
        return self.layer(x)

correct_model = CorrectModel()

print(correct_model)


# 5. Understanding `forward()`

The `forward()` method defines how data moves through the model.

For our model:

```python
def forward(self, x):
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    return x
```

Conceptually:

$$
\text{Input}
\rightarrow
fc1
\rightarrow
ReLU
\rightarrow
fc2
\rightarrow
\text{Output}
$$

`forward()` defines computation.

`__init__()` defines structure.


# 6. Why We Call `model(x)` Instead of `model.forward(x)`

In normal PyTorch code, use:

```python
output = model(x)
```

instead of:

```python
output = model.forward(x)
```

Calling `model(x)` lets `nn.Module` run additional internal logic around the forward pass.

This includes support for:

- Hooks
- Wrappers
- Framework internals

So the recommended pattern is:

> **Define `forward()`, but call the model using `model(x)`.**


In [ ]:
x = torch.randn(5, 3)

output = model(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)


# 7. Registered Parameters

A parameter is a tensor that PyTorch recognizes as trainable model state.

Examples:

- Linear-layer weights
- Linear-layer biases
- Convolution kernels

For:

`nn.Linear(3,4)`

the weight shape is:

$$
\boxed{(4,\ 3)}
$$

and the bias shape is:

$$
\boxed{(4)}
$$


In [ ]:
print("fc1 weight shape:", model.fc1.weight.shape)
print("fc1 bias shape:", model.fc1.bias.shape)

print("fc2 weight shape:", model.fc2.weight.shape)
print("fc2 bias shape:", model.fc2.bias.shape)


# 8. What Is `nn.Parameter`?

`nn.Parameter` is a special kind of tensor.

When an `nn.Parameter` is assigned as an attribute of an `nn.Module`, PyTorch automatically registers it as a model parameter.


In [ ]:
parameter = nn.Parameter(torch.tensor([1.0, 2.0, 3.0]))

print(parameter)
print("requires_grad:", parameter.requires_grad)


Notice that `nn.Parameter` normally has:

`requires_grad=True`

because it is intended to represent learnable model state.


# 9. Manual Parameter Registration

Let's build a simple custom linear transformation without using `nn.Linear`.

We want:

$$
y=xW^T+b
$$


In [ ]:
class ManualLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        self.weight = nn.Parameter(
            torch.randn(out_features, in_features)
        )

        self.bias = nn.Parameter(
            torch.zeros(out_features)
        )

    def forward(self, x):
        return x @ self.weight.T + self.bias

manual_linear = ManualLinear(3, 2)

print(manual_linear)


In [ ]:
for name, parameter in manual_linear.named_parameters():
    print(name, parameter.shape)


Because `weight` and `bias` are `nn.Parameter` objects, PyTorch automatically finds them.

This is exactly the type of mechanism used inside built-in layers.


# 10. Ordinary Tensor vs `nn.Parameter`

Consider this model:


In [ ]:
class TensorVsParameter(nn.Module):
    def __init__(self):
        super().__init__()

        self.normal_tensor = torch.randn(3)
        self.learnable_tensor = nn.Parameter(torch.randn(3))

    def forward(self, x):
        return x

example_model = TensorVsParameter()

print(example_model)


In [ ]:
print("Named parameters:")

for name, parameter in example_model.named_parameters():
    print(name)


Only:

`learnable_tensor`

appears in the parameter list.

The ordinary tensor:

`normal_tensor`

is not automatically treated as a trainable parameter.

This distinction is important.


# 11. `model.parameters()`

`model.parameters()` returns an iterator over registered model parameters.

This is commonly passed to an optimizer:

```python
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)
```


In [ ]:
for parameter in model.parameters():
    print(parameter.shape)


# 12. Counting Parameters

We can count the total number of scalar parameters using:

`parameter.numel()`


In [ ]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Total parameters:", total_parameters)


For our model:

$$
3\rightarrow4\rightarrow2
$$

First layer:

$$
4\times3+4=16
$$

Second layer:

$$
2\times4+2=10
$$

Total:

$$
\boxed{26}
$$


# 13. `model.named_parameters()`

`model.named_parameters()` returns both:

- Parameter name
- Parameter tensor

This is extremely useful for debugging.


In [ ]:
for name, parameter in model.named_parameters():
    print(
        name,
        "| shape:",
        tuple(parameter.shape),
        "| requires_grad:",
        parameter.requires_grad
    )


Typical names look like:

- `fc1.weight`
- `fc1.bias`
- `fc2.weight`
- `fc2.bias`

The names reflect the module hierarchy.


# 14. Parameter Hierarchy

Suppose a model contains a layer called:

`fc1`

and that layer contains:

- `weight`
- `bias`

Then the parameter names become:

$$
\boxed{fc1.weight}
$$

and:

$$
\boxed{fc1.bias}
$$

This hierarchical naming becomes very useful in large models.


# 15. Submodules

A model can contain other `nn.Module` objects.

These are called **submodules**.

Our `TinyModel` contains:

- `nn.Linear`
- `nn.ReLU`
- `nn.Linear`

We can inspect them using:

`model.named_modules()`


In [ ]:
for name, module in model.named_modules():
    print(f"{name!r}: {module.__class__.__name__}")


The empty name:

`''`

represents the top-level model itself.

The other entries represent submodules.


# 16. `model.children()`

`model.children()` returns the immediate child modules.


In [ ]:
for child in model.children():
    print(child)


# 17. `model.modules()`

`model.modules()` recursively includes:

- The model itself
- Child modules
- Nested child modules


In [ ]:
for module in model.modules():
    print(type(module).__name__)


# 18. Nested Models

Large neural networks are often built from smaller reusable modules.

Let's create a reusable block.


In [ ]:
class HiddenBlock(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.activation(self.linear(x))

block = HiddenBlock(3, 4)

print(block)


Now we can reuse this block inside a larger model.


In [ ]:
class BlockNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = HiddenBlock(3, 8)
        self.block2 = HiddenBlock(8, 4)
        self.output = nn.Linear(4, 2)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x

block_model = BlockNetwork()

print(block_model)


# 19. Shape Reasoning Through Nested Modules

For input:

$$
X.shape=(16,\ 3)
$$

the model performs:

$$
(16,\ 3)
\rightarrow
(16,\ 8)
\rightarrow
(16,\ 4)
\rightarrow
(16,\ 2)
$$


In [ ]:
X = torch.randn(16, 3)

output = block_model(X)

print("Input shape:", X.shape)
print("Output shape:", output.shape)


# 20. `nn.Sequential`

`nn.Sequential` is useful when layers are applied one after another in a simple fixed order.

Example:

$$
3\rightarrow8\rightarrow4\rightarrow2
$$


In [ ]:
sequential_model = nn.Sequential(
    nn.Linear(3, 8),
    nn.ReLU(),
    nn.Linear(8, 4),
    nn.ReLU(),
    nn.Linear(4, 2)
)

print(sequential_model)


In [ ]:
X = torch.randn(16, 3)

output = sequential_model(X)

print("Input shape:", X.shape)
print("Output shape:", output.shape)


# 21. Named Layers Inside `nn.Sequential`

We can create named layers using `OrderedDict`.


In [ ]:
from collections import OrderedDict

named_sequential = nn.Sequential(
    OrderedDict([
        ("fc1", nn.Linear(3, 8)),
        ("relu1", nn.ReLU()),
        ("fc2", nn.Linear(8, 4)),
        ("relu2", nn.ReLU()),
        ("output", nn.Linear(4, 2)),
    ])
)

print(named_sequential)


Named layers can make inspection easier.


In [ ]:
print(named_sequential.fc1)
print(named_sequential.output)


# 22. `nn.Sequential` vs Custom `nn.Module`

$$
\begin{array}{|c|c|}
\hline
\textbf{nn.Sequential} & \textbf{Custom nn.Module} \\
\hline
\text{Concise} & \text{More flexible} \\
\hline
\text{Simple ordered pipeline} & \text{Custom forward logic} \\
\hline
\text{Great for straightforward stacks} & \text{Useful for branches and skips} \\
\hline
\end{array}
$$

Use `nn.Sequential` when the architecture is simply:

$$
Layer_1
\rightarrow
Layer_2
\rightarrow
Layer_3
$$

Use a custom `nn.Module` when forward logic is more complex.


# 23. Custom Layers

We can create our own layer by subclassing `nn.Module`.

Let's create a layer that learns a scale and bias:

$$
y=ax+b
$$

where `a` and `b` are learnable parameters.


In [ ]:
class LearnableScaleShift(nn.Module):
    def __init__(self, features):
        super().__init__()

        self.scale = nn.Parameter(torch.ones(features))
        self.shift = nn.Parameter(torch.zeros(features))

    def forward(self, x):
        return self.scale * x + self.shift

custom_layer = LearnableScaleShift(3)

print(custom_layer)


In [ ]:
X = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])

output = custom_layer(X)

print("Output:")
print(output)


# 24. Custom-Layer Parameters

The custom layer contains:

$$
scale.shape=(3)
$$

and:

$$
shift.shape=(3)
$$

Both are automatically registered because they are `nn.Parameter` objects.


In [ ]:
for name, parameter in custom_layer.named_parameters():
    print(name, parameter.shape)


# 25. `state_dict()`

A model's `state_dict()` contains its persistent state.

For most models, this includes:

- Parameters
- Registered buffers

For our simple model, it contains weights and biases.


In [ ]:
state = model.state_dict()

print(state.keys())


Typical keys are:

- `fc1.weight`
- `fc1.bias`
- `fc2.weight`
- `fc2.bias`

The keys match the module hierarchy.


In [ ]:
for key, value in model.state_dict().items():
    print(key, tuple(value.shape))


# 26. `state_dict()` vs `parameters()`

`model.parameters()` gives trainable parameter tensors.

`model.state_dict()` gives named persistent model state.

A useful distinction:

$$
\begin{array}{|c|c|}
\hline
\textbf{parameters()} & \textbf{state_dict()} \\
\hline
\text{Iterator of parameters} & \text{Dictionary-like mapping} \\
\hline
\text{Used by optimizers} & \text{Used for saving/loading} \\
\hline
\text{No names by default} & \text{Contains names} \\
\hline
\end{array}
$$


# 27. Registered Buffers

Some model state should be saved and moved between devices but should **not** be optimized.

For this, PyTorch supports:

`register_buffer()`

A buffer is not a trainable parameter.


In [ ]:
class ModelWithBuffer(nn.Module):
    def __init__(self):
        super().__init__()

        self.weight = nn.Parameter(torch.tensor(2.0))

        self.register_buffer(
            "running_value",
            torch.tensor(1.0)
        )

    def forward(self, x):
        return x * self.weight + self.running_value

buffer_model = ModelWithBuffer()

print(buffer_model)


In [ ]:
print("Parameters:")
for name, parameter in buffer_model.named_parameters():
    print(name, parameter)

print("\\nBuffers:")
for name, buffer in buffer_model.named_buffers():
    print(name, buffer)


The buffer appears inside the model state:


In [ ]:
print(buffer_model.state_dict())


This idea is used internally by layers such as Batch Normalization.


# 28. Training Mode vs Evaluation Mode

Every `nn.Module` has a mode:

- Training mode
- Evaluation mode

Use:

`model.train()`

for training.

Use:

`model.eval()`

for evaluation.


In [ ]:
model.train()

print("Training mode:", model.training)

model.eval()

print("Training mode after eval():", model.training)


# 29. Why Training and Evaluation Modes Matter

Some layers behave differently during training and evaluation.

Important examples:

- `nn.Dropout`
- Batch Normalization layers

For ordinary `nn.Linear` and `nn.ReLU`, the behavior is usually the same.

But for models containing Dropout or BatchNorm, calling:

`model.eval()`

during validation and testing is essential.


# 30. Dropout Example

Dropout randomly sets some activations to zero during training.

This randomness is disabled during evaluation.


In [ ]:
torch.manual_seed(42)

dropout = nn.Dropout(p=0.5)

x = torch.ones(10)

dropout.train()
training_output = dropout(x)

dropout.eval()
evaluation_output = dropout(x)

print("Input:")
print(x)

print("\\nTraining output:")
print(training_output)

print("\\nEvaluation output:")
print(evaluation_output)


During training, some values are zeroed.

During evaluation, Dropout acts like an identity transformation.

This is why model mode matters.


# 31. Model Mode Propagates to Submodules

Calling:

`model.train()`

or:

`model.eval()`

updates the mode recursively for child modules.


In [ ]:
dropout_model = nn.Sequential(
    nn.Linear(3, 4),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(4, 2)
)

dropout_model.eval()

for name, module in dropout_model.named_modules():
    print(
        name if name else "<root>",
        "| training:",
        module.training
    )


# 32. `eval()` Does Not Disable Gradients

This is a very important distinction:

`model.eval()`

does **not** turn off Autograd.

It only changes module behavior.

To disable gradient tracking during inference, commonly use:

```python
model.eval()

with torch.no_grad():
    predictions = model(x)
```


In [ ]:
model.eval()

x = torch.randn(2, 3, requires_grad=True)

output = model(x)

print("Grad tracking without no_grad:", output.requires_grad)

with torch.no_grad():
    output_no_grad = model(x)

print("Grad tracking with no_grad:", output_no_grad.requires_grad)


# 33. Saving Model Weights

The recommended basic pattern is:

```python
torch.save(
    model.state_dict(),
    "model_weights.pth"
)
```

This saves the model parameters and persistent buffers.


In [ ]:
weights_path = "tiny_model_weights.pth"

torch.save(
    model.state_dict(),
    weights_path
)

print("Saved to:", weights_path)


# 34. Loading Model Weights

To load weights:

1. Recreate the model architecture
2. Load the state dictionary


In [ ]:
loaded_model = TinyModel()

loaded_state = torch.load(
    weights_path,
    weights_only=True
)

loaded_model.load_state_dict(loaded_state)

print(loaded_model)


# 35. Verifying Saved and Loaded Models

If the architecture and weights are identical, the models should produce identical outputs for the same input.


In [ ]:
model.eval()
loaded_model.eval()

x = torch.randn(5, 3)

with torch.no_grad():
    original_output = model(x)
    loaded_output = loaded_model(x)

print(
    "Outputs match:",
    torch.allclose(
        original_output,
        loaded_output
    )
)


# 36. Why Save `state_dict()` Instead of the Whole Model?

Saving the `state_dict()` is often preferred because it is:

- Explicit
- Flexible
- Easy to inspect
- Less dependent on pickling the complete Python object

The architecture is defined in code.

The weights are stored separately.

This separation is a strong PyTorch practice.


# 37. Strict Loading

By default:

`load_state_dict()`

expects model-state keys to match.

This helps catch architectural mismatches.


In [ ]:
fresh_model = TinyModel()

result = fresh_model.load_state_dict(
    model.state_dict()
)

print(result)


# 38. Building Reusable Model Classes

A reusable model class should make important architecture choices configurable.

Instead of hardcoding:

$$
3\rightarrow4\rightarrow2
$$

we can accept:

- Input size
- Hidden size
- Output size


In [ ]:
class FlexibleMLP(nn.Module):
    def __init__(
        self,
        in_features,
        hidden_features,
        out_features
    ):
        super().__init__()

        self.fc1 = nn.Linear(
            in_features,
            hidden_features
        )

        self.relu = nn.ReLU()

        self.fc2 = nn.Linear(
            hidden_features,
            out_features
        )

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

flexible_model = FlexibleMLP(
    in_features=10,
    hidden_features=32,
    out_features=4
)

print(flexible_model)


In [ ]:
X = torch.randn(8, 10)

output = flexible_model(X)

print("Input shape:", X.shape)
print("Output shape:", output.shape)


# 39. Storing Architecture Information

Sometimes it is helpful to store configuration as attributes.


In [ ]:
class ConfigurableMLP(nn.Module):
    def __init__(
        self,
        in_features,
        hidden_features,
        out_features
    ):
        super().__init__()

        self.in_features = in_features
        self.hidden_features = hidden_features
        self.out_features = out_features

        self.network = nn.Sequential(
            nn.Linear(
                in_features,
                hidden_features
            ),
            nn.ReLU(),
            nn.Linear(
                hidden_features,
                out_features
            )
        )

    def forward(self, x):
        return self.network(x)

config_model = ConfigurableMLP(
    5,
    16,
    3
)

print(config_model)


# 40. Device Movement With `nn.Module`

A major benefit of registered parameters and buffers is that PyTorch can move them together.

Example:

```python
model = model.to(device)
```


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

device_model = TinyModel().to(device)

print("Device:", device)

for name, parameter in device_model.named_parameters():
    print(name, parameter.device)


Inputs must normally be moved to the same device.


In [ ]:
x = torch.randn(4, 3).to(device)

output = device_model(x)

print("Input device:", x.device)
print("Output device:", output.device)


# 41. Why Registration Matters for Device Movement

PyTorch can automatically move:

- Registered parameters
- Registered buffers
- Registered submodules

But arbitrary tensors stored outside these mechanisms may require manual handling.

This is another reason to use:

- `nn.Parameter`
- `register_buffer()`
- Proper submodule assignment


# 42. Freezing Parameters

Sometimes we do not want a parameter to be trained.

We can set:

`requires_grad=False`


In [ ]:
freeze_model = TinyModel()

for parameter in freeze_model.fc1.parameters():
    parameter.requires_grad = False

for name, parameter in freeze_model.named_parameters():
    print(
        name,
        "| requires_grad:",
        parameter.requires_grad
    )


This is important later for:

- Transfer learning
- Feature extraction
- Fine-tuning pretrained models


# 43. Counting Only Trainable Parameters


In [ ]:
total = sum(
    p.numel()
    for p in freeze_model.parameters()
)

trainable = sum(
    p.numel()
    for p in freeze_model.parameters()
    if p.requires_grad
)

print("Total parameters:", total)
print("Trainable parameters:", trainable)


# 44. Inspecting a Model Before Training

Before training a new model, useful checks include:

1. Print the model
2. Check input shape
3. Run one forward pass
4. Check output shape
5. Inspect parameter shapes
6. Count trainable parameters
7. Confirm device placement


In [ ]:
debug_model = FlexibleMLP(
    in_features=6,
    hidden_features=12,
    out_features=2
)

sample_batch = torch.randn(4, 6)

print(debug_model)

print("\\nInput:", sample_batch.shape)

output = debug_model(sample_batch)

print("Output:", output.shape)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in debug_model.parameters()
        if p.requires_grad
    )
)


# 45. Common `nn.Module` Mistakes

## Mistake 1 — Forgetting `super().__init__()`

This can break module and parameter registration.

## Mistake 2 — Creating Layers Inside `forward()`

Trainable layers should usually be created in `__init__()`.

Creating a new `nn.Linear` inside every forward pass creates fresh parameters repeatedly.

## Mistake 3 — Using Plain Tensors for Learnable Parameters

Use `nn.Parameter` when a custom tensor should be trainable.

## Mistake 4 — Calling `forward()` Directly

Normally call:

`model(x)`

## Mistake 5 — Forgetting `model.eval()`

This can produce incorrect evaluation behavior with Dropout or BatchNorm.

## Mistake 6 — Thinking `eval()` Disables Gradients

It does not.

Use `torch.no_grad()` when gradient tracking is unnecessary.

## Mistake 7 — Loading Weights Into the Wrong Architecture

The model structure must be compatible with the saved `state_dict()`.

## Mistake 8 — Forgetting Device Alignment

Model and inputs should usually be on the same device.


# 46. Example of a Bad Pattern

This is a bad model design:

```python
class BadModel(nn.Module):
    def forward(self, x):
        layer = nn.Linear(3, 2)
        return layer(x)
```

Why?

Every forward pass creates a **new layer with new random parameters**.

Those parameters are not persistent model state in the intended way.

Layers should normally be created in `__init__()`.


# 47. Correct Pattern


In [ ]:
class GoodModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer = nn.Linear(3, 2)

    def forward(self, x):
        return self.layer(x)

good_model = GoodModel()

print(good_model)


# 48. Model Structure Debugging Checklist

When a model is not behaving as expected, check:

- Does the class inherit from `nn.Module`?
- Did you call `super().__init__()`?
- Are trainable layers created in `__init__()`?
- Are custom trainable tensors `nn.Parameter` objects?
- Do `named_parameters()` show what you expect?
- Does `state_dict()` contain the expected keys?
- Is the model in the correct train/eval mode?
- Are model and inputs on the same device?
- Does the forward pass preserve expected shapes?
- Does the loaded architecture match saved weights?


# 49. Practice Exercises

Try these before looking at the solutions.

## Exercise 1

Build a model:

$$
4\rightarrow8\rightarrow2
$$

using a custom `nn.Module`.

## Exercise 2

Print all parameter names and shapes.

## Exercise 3

Count total parameters.

## Exercise 4

Rebuild the same model using `nn.Sequential`.

## Exercise 5

Create a custom layer with a learnable scalar:

$$
y=ax
$$

where $a$ is an `nn.Parameter`.

## Exercise 6

Create a model containing a registered buffer.

## Exercise 7

Print the model's `state_dict()` keys.

## Exercise 8

Save and reload the model's weights.

## Exercise 9

Add Dropout and compare behavior in:

- `train()` mode
- `eval()` mode

## Exercise 10

Freeze the first layer and count trainable parameters again.


# 50. Shape Reasoning Challenges

Answer before running code.

## Challenge 1

For:

`nn.Linear(10, 20)`

what are:

- Weight shape
- Bias shape

## Challenge 2

Input:

$$
(32,\ 10)
$$

through:

`nn.Linear(10,20)`

gives what output shape?

## Challenge 3

Model:

$$
10\rightarrow20\rightarrow5
$$

Batch size:

$$
64
$$

Write the shape after each layer.

## Challenge 4

How many parameters are in:

`nn.Linear(10,20)`?

## Challenge 5

Why does a registered buffer appear in `state_dict()` but not in `parameters()`?

## Challenge 6

Why is `model.eval()` not the same as `torch.no_grad()`?


# 51. Exercise Solutions


In [ ]:
# Exercise 1
class ExerciseModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(4, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.fc2(x)

exercise_model = ExerciseModel()

print("Exercise 1:")
print(exercise_model)

# Exercise 2
print("\\nExercise 2:")
for name, parameter in exercise_model.named_parameters():
    print(name, parameter.shape)

# Exercise 3
print(
    "\\nExercise 3:",
    sum(
        p.numel()
        for p in exercise_model.parameters()
    )
)

# Exercise 4
exercise_sequential = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

print("\\nExercise 4:")
print(exercise_sequential)

# Exercise 5
class LearnableMultiplier(nn.Module):
    def __init__(self):
        super().__init__()
        self.a = nn.Parameter(torch.tensor(1.0))

    def forward(self, x):
        return self.a * x

multiplier = LearnableMultiplier()

print("\\nExercise 5 parameter:")
print(list(multiplier.named_parameters()))

# Exercise 6
class BufferExercise(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer(
            "constant",
            torch.tensor(2.0)
        )

    def forward(self, x):
        return x + self.constant

buffer_exercise = BufferExercise()

print("\\nExercise 6 buffers:")
print(list(buffer_exercise.named_buffers()))

# Exercise 7
print("\\nExercise 7:")
print(exercise_model.state_dict().keys())

# Exercise 8
exercise_path = "exercise_model_weights.pth"

torch.save(
    exercise_model.state_dict(),
    exercise_path
)

reloaded_exercise_model = ExerciseModel()

reloaded_exercise_model.load_state_dict(
    torch.load(
        exercise_path,
        weights_only=True
    )
)

print("\\nExercise 8: weights loaded")

# Exercise 9
drop = nn.Dropout(0.5)
sample = torch.ones(8)

drop.train()
print("\\nExercise 9 train:", drop(sample))

drop.eval()
print("Exercise 9 eval :", drop(sample))

# Exercise 10
for parameter in exercise_model.fc1.parameters():
    parameter.requires_grad = False

print(
    "\\nExercise 10 trainable:",
    sum(
        p.numel()
        for p in exercise_model.parameters()
        if p.requires_grad
    )
)


# 52. Key Takeaways

In this notebook, we learned:

- `nn.Module`
- `__init__()`
- `forward()`
- Why `super().__init__()` matters
- Automatic submodule registration
- `nn.Parameter`
- Parameter registration
- `model.parameters()`
- `model.named_parameters()`
- Parameter counting
- Submodules
- `named_modules()`
- `children()`
- Nested modules
- `nn.Sequential`
- Custom layers
- `state_dict()`
- Registered buffers
- `train()` vs `eval()`
- Dropout behavior
- `eval()` vs `torch.no_grad()`
- Saving model weights
- Loading model weights
- Reusable model classes
- Device movement
- Freezing parameters

The central structure of a PyTorch model is:

$$
\boxed{
\text{Registered Modules}
+
\text{Registered Parameters}
+
\text{Forward Logic}
}
$$


# 53. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. What is `nn.Module`?
2. What belongs in `__init__()`?
3. What belongs in `forward()`?
4. Why is `super().__init__()` important?
5. Why do we call `model(x)`?
6. What is `nn.Parameter`?
7. What is parameter registration?
8. What does `model.parameters()` return?
9. What does `model.named_parameters()` return?
10. What is a submodule?
11. What does `state_dict()` contain?
12. What is a registered buffer?
13. Why would a buffer not appear in `parameters()`?
14. What does `model.train()` do?
15. What does `model.eval()` do?
16. Why is `eval()` different from `no_grad()`?
17. What is the recommended basic way to save model weights?
18. How do you load a `state_dict()`?
19. Why should trainable layers usually be created in `__init__()`?
20. Why are reusable model classes useful?


# Next Notebook

# 10 — Loss Functions

In the next notebook, we will study:

- What is a loss function?
- Regression vs classification losses
- Mean Squared Error
- Mean Absolute Error
- Binary Cross Entropy
- `BCEWithLogitsLoss`
- Sigmoid and logits
- Multi-class classification
- `CrossEntropyLoss`
- Softmax intuition
- Target shapes and dtypes
- Choosing the correct loss
- Common loss-function mistakes
